# Clase 17 · Backpropagation manual

**Identificador técnico:** `16_backpropagation_manual`  
**Dataset real:** iris  
**Fuente:** UCI  
**Versión:** solución

## Antes de ejecutar

La salida conoce el error, pero las primeras capas están lejos de él. La regla de la cadena transporta esa señal hacia atrás.

> **Pregunta esencial:** ¿Cómo se reparte la responsabilidad del error entre todas las capas?

Escribe una predicción breve. No busques acertar: la recuperarás al final para explicar qué evidencia cambió tu idea.

![La responsabilidad viaja hacia atrás](assets/class-map.svg)

**Cómo leer el mapa:** Error de salida → Regla de la cadena → Gradiente por parámetro.

## Objetivo

Derivar y programar backpropagation en una MLP pequeña.

**Práctica propia:** Derivar una MLP capa por capa, comprobar el gradiente y graficar su norma a distintas profundidades.

## Contrato científico

1. `split_seed` define las particiones.  
2. `training_seed` define inicialización y batches.  
3. Toda selección ocurre con validation.  
4. `experiment.lock.json` congela decisiones antes de abrir test.  
5. Test se informa una sola vez.

In [ ]:
from pathlib import Path
import json
import yaml
import torch

from neural_labs.catalog import ROOT, get_lab, get_dataset
from neural_labs.datasets import prepare_dataset, audit_bundle
from neural_labs.baselines import run_baseline
from neural_labs.experiments import run_lab

LAB_ID = '16_backpropagation_manual'
SPLIT_SEED = 42
TRAINING_SEED = 42
QUICK = True

In [ ]:
lab = get_lab(LAB_ID)
dataset_card = get_dataset(LAB_ID)
lab, dataset_card

## Procedencia, licencia y limitaciones

Lee la ficha antes de descargar. No redistribuyas datos si la licencia no lo permite.

In [ ]:
manifest_path = ROOT / 'labs' / LAB_ID / 'data' / 'dataset.yaml'
manifest = yaml.safe_load(manifest_path.read_text(encoding='utf-8'))
manifest

## Preparación y auditoría

La siguiente celda descarga datos reales. Puede requerir Internet, credenciales de Kaggle o aceptación de condiciones.

In [ ]:
# bundle = prepare_dataset(LAB_ID, quick=QUICK, seed=SPLIT_SEED)
# audit_bundle(bundle)
print('Descomenta para descargar y preparar el dataset real.')

## Exploración específica del dominio: tabular

Compara contra regresión logística/árboles, revisa calibración, subgrupos, importancia de variables y estabilidad.

In [ ]:
# Inspección de atributos y distribución de clases
# print(bundle.feature_names[:20])
# print(bundle.summary)

## Línea base sobre validation

La línea base no debe mirar test antes de congelar el experimento.

In [ ]:
# baseline_validation = run_baseline(LAB_ID, bundle, quick=QUICK, evaluation_split='validation')
# baseline_validation

## Modelo y teoría

**Arquitectura:** `numpy_mlp`  
**Fundamento:** Regla de la cadena para W2, b2, W1 y b1.

In [ ]:
# sample, target = bundle.train[0]
# print('input:', sample.shape, 'target:', target)
# print('selection metric:', lab['selection_metric'])

## Entrenamiento reproducible

El comando guarda contrato de inferencia, lock experimental, métricas, checkpoint y tarjetas.

In [ ]:
# result = run_lab(
#     LAB_ID, quick=QUICK, config_name='baseline',
#     split_seed=SPLIT_SEED, training_seed=TRAINING_SEED, device='auto',
# )
# result.run_dir, result.metrics

## Inspección de artefactos

In [ ]:
# sorted(path.name for path in result.run_dir.iterdir())

## Interpretación y responsabilidad

Documenta resultados negativos, sesgos, grupos con peor desempeño, costo computacional y usos no recomendados.

> **Error conceptual que debes poder detectar:** Un programa que ejecuta backward sin error puede seguir teniendo gradientes matemáticamente incorrectos.

## Actividad propia de esta clase

Derivar una MLP capa por capa, comprobar el gradiente y graficar su norma a distintas profundidades.

Cierra la actividad volviendo a tu predicción inicial: indica qué observaste, qué explicación propones y qué nueva prueba harías.

## Ejercicios evaluables

Cinco ejercicios sobre el contrato experimental de este laboratorio. Se resuelven con Python estándar: **no hace falta descargar el dataset ni entrenar**, y cada uno trae debajo una celda de comprobación que debe pasar sin error.

### Ejercicio 1 — Auditar la partición

Antes de entrenar hay que demostrar que ningún ejemplo aparece en dos particiones. Una sola fila compartida entre `train` y `test` infla la métrica final sin dar ningún síntoma.

Escribe `sin_solapamiento(train_ids, validation_ids, test_ids)`, que devuelve `True` solo si los tres conjuntos de identificadores son disjuntos dos a dos.

In [ ]:
# SOLUCIÓN DE REFERENCIA
def sin_solapamiento(train_ids, validation_ids, test_ids):
    train, validation, test = set(train_ids), set(validation_ids), set(test_ids)
    return not (train & validation or train & test or validation & test)

In [ ]:
assert sin_solapamiento([1, 2, 3], [4, 5], [6]) is True
assert sin_solapamiento([1, 2, 3], [3, 4], [6]) is False   # solapa train y validation
assert sin_solapamiento([1, 2], [4], [2]) is False         # solapa train y test
assert sin_solapamiento([1], [2, 3], [3]) is False         # solapa validation y test
assert sin_solapamiento([], [], []) is True
print('Auditoría correcta.')

### Ejercicio 2 — Elegir el checkpoint con `validation`

Este laboratorio selecciona con **`macro_f1`**, donde **mayor es mejor**. El modelo que se conserva es el de la época con mejor valor *en validación*, nunca en test.

Escribe `mejor_epoca(historial)`, que recibe una lista de diccionarios con las claves `epoch` y `macro_f1` y devuelve el número de la mejor época.

In [ ]:
# SOLUCIÓN DE REFERENCIA
SELECTION_METRIC = 'macro_f1'
HIGHER_IS_BETTER = True

def mejor_epoca(historial):
    elegir = max if HIGHER_IS_BETTER else min
    mejor = elegir(historial, key=lambda fila: fila[SELECTION_METRIC])
    return mejor['epoch']

In [ ]:
historial = [
    {'epoch': 1, 'macro_f1': 0.40},
    {'epoch': 2, 'macro_f1': 0.65},
    {'epoch': 3, 'macro_f1': 0.55},
]
assert HIGHER_IS_BETTER is True
assert mejor_epoca(historial) == 2
print('Regla de selección correcta.')

### Ejercicio 3 — Comparar contra la línea base

La línea base de este laboratorio es **Regresión logística multinomial**. Superarla por poco no basta: la diferencia tiene que ser mayor que la dispersión del propio modelo entre semillas, o no se puede distinguir de la suerte.

Escribe `supera_linea_base(puntajes, base)`, que recibe la lista de puntajes del modelo (uno por semilla) y el puntaje de la línea base, y devuelve un diccionario con `media`, `desviacion` y `concluyente`.

In [ ]:
# SOLUCIÓN DE REFERENCIA
from statistics import mean, pstdev

def supera_linea_base(puntajes, base):
    media = mean(puntajes)
    desviacion = pstdev(puntajes) if len(puntajes) > 1 else 0.0
    ventaja = media - base if True else base - media
    return {
        'media': media,
        'desviacion': desviacion,
        'concluyente': ventaja > desviacion,
    }

In [ ]:
claro = supera_linea_base([0.80, 0.81, 0.82], 0.60)
dudoso = supera_linea_base([0.55, 0.65, 0.75], 0.60)
assert round(claro['media'], 4) == 0.81
assert claro['concluyente'] is True
assert dudoso['concluyente'] is False   # la mejora cabe dentro del ruido
print('Comparación correcta:', claro)

### Ejercicio 4 — No abrir `test` sin sellar

El repositorio escribe `experiment.lock.json` con las semillas, la configuración y el checkpoint elegido **antes** de evaluar `test`. Sin ese archivo, cualquier número de test es inválido.

Escribe `puede_abrir_test(run_dir)`, que devuelve `True` solo si el directorio de la ejecución contiene `experiment.lock.json`.

In [ ]:
# SOLUCIÓN DE REFERENCIA
from pathlib import Path

def puede_abrir_test(run_dir):
    return (Path(run_dir) / 'experiment.lock.json').is_file()

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as carpeta:
    ruta = Path(carpeta)
    assert puede_abrir_test(ruta) is False   # todavía no se selló
    (ruta / 'experiment.lock.json').write_text('{}', encoding='utf-8')
    assert puede_abrir_test(ruta) is True
print('Sellado comprobado.')

### Ejercicio 5 — Dejar el plan por escrito

El experimento propio de esta ruta es: **Validar derivadas capa por capa**.

Declara el plan antes de ejecutarlo: qué hipótesis pones a prueba, qué variable cambias, qué mantienes fijo, con qué semillas de entrenamiento y qué conclusión esperas poder escribir. Completa las cinco variables.

In [ ]:
# SOLUCIÓN DE REFERENCIA
HIPOTESIS = 'Validar derivadas capa por capa mejora macro_f1 frente a Regresión logística multinomial.'
VARIABLE_QUE_CAMBIA = 'Validar derivadas capa por capa'
VARIABLES_CONTROLADAS = ['misma partición (split_seed=42)', 'mismo presupuesto de épocas', 'mismo dataset: iris']
SEMILLAS_DE_ENTRENAMIENTO = [41, 42, 43]
CONCLUSION = ('La diferencia solo es concluyente si supera la dispersión entre semillas; si no, se reporta que no se distingue del ruido, y se compara siempre contra Regresión logística multinomial.')

In [ ]:
assert len(HIPOTESIS) >= 30, 'La hipótesis debe poder resultar falsa; descríbela.'
assert len(VARIABLE_QUE_CAMBIA) >= 10, 'Un experimento cambia una cosa a la vez.'
assert len(VARIABLES_CONTROLADAS) >= 3, 'Enumera al menos tres variables controladas.'
assert len(SEMILLAS_DE_ENTRENAMIENTO) >= 3, 'Con menos de tres semillas no hay dispersión.'
assert len(CONCLUSION) >= 40, 'La conclusión debe hablar de magnitud e incertidumbre.'
print('Plan experimental completo.')

## Próximos pasos

Ejecuta primero con `--quick`; después usa el dataset completo, varias semillas y el pipeline de benchmark.